In [1]:
# Adapted from Inverted Pendulum MPC Control: Atsushi Sakai (Python Robotics)
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import train_test_split
import numpy as np
import time
import math
from tqdm.notebook import tqdm
import imageio

In [2]:
%matplotlib

Using matplotlib backend: Qt5Agg


Reset function should always start at same state

## Inverted Pendulum Environment

In [3]:
def reset():
    pass

def angle_normalize(x):
    return (((x+np.pi) % (2*np.pi)) - np.pi)

In [4]:
angle_normalize(0.5)

0.5

Plotting function, for animation

In [30]:
def plot_pendulum(theta,theta_dot):

    radius = 0.1
    scale = 1.5

    bx = np.array([0.0, l* math.sin(-theta)])
    by = np.array([0.0, l* math.cos(-theta)])

    angles = np.arange(0.0, math.pi * 2.0, math.radians(3.0))
    
    ox = np.array([radius * math.cos(a) for a in angles])
    oy = np.array([radius * math.sin(a) for a in angles])

    rwx = np.copy(ox)/scale 
    rwy = np.copy(oy)/scale

    # pendulum bob
    wx = scale*np.copy(ox) + bx[-1]
    wy = scale*np.copy(oy) + by[-1]
    plt.plot(flatten(wx), flatten(wy), "-r")
    
    
    # pendulum axle
    plt.plot(flatten(bx), flatten(by), "-k")
    
    # anchor
    plt.plot(flatten(rwx), flatten(rwy), "-k")

    plt.title("$\Theta$:" +
              str(round(theta, 2)) + "   $\dot{\Theta }$:" + str(round(theta_dot, 2)))

    plt.axis("equal")

# Build GIF
def build_gif(imagepath=''):
    with imageio.get_writer('mygif.gif', mode='I') as writer:
        for f in range(50):
            
            filename=imagepath+str(f)+'.png'
            image = imageio.imread(filename)
            writer.append_data(image)
def flatten(a):
    return np.array(a).flatten()

Dynamics function

In [31]:
max_speed=1e6
max_torque=2.
dt=.07 # sampling period: (1/Fs)
g = 10.
m = 1.
l = 1. # length of bar
        
def step(state,u):
    th = state[0] # th := theta
    thdot = state[1] 

    # u = np.clip(u, -max_torque, max_torque)
    
    costs = angle_normalize(th)**2 + .1*thdot**2 + .001*(u**2)
    
    newthdot = thdot + (-3*g/(2*l) * np.sin(th + np.pi) + 3./(m*l**2)*u) * dt
    newth = th + newthdot*dt
    # newthdot = np.clip(newthdot, max_speed, max_speed) # pylint: disable=E1111
    state = np.array([newth, newthdot])
    return state, -costs, False, {}

In [32]:
def animate(x,i):
    plt.clf()
    px = 1
    theta = 0
    plot_pendulum(x[0],x[1])
    plt.xlim([-2, 2])
    plt.ylim([-2, 2])
    plt.pause(0.0001)
    plt.savefig(str(i)+'.png')

## Linear Regression Model
The maximum likelihood estimator is given by
$$
\boldsymbol\theta^{\text{ML}} = (\boldsymbol X^T\boldsymbol X)^{-1}\boldsymbol X^T\boldsymbol y\in\mathbb{R}^D\,,
$$
where 
$$
\boldsymbol X = [\boldsymbol x_1, \ldots, \boldsymbol x_N]^T\in\mathbb{R}^{N\times D}\,,\quad \boldsymbol y = [y_1, \ldots, y_N]^T \in\mathbb{R}^N\,.
$$

In [33]:
def lr_ml_estimate(X, y):
    
    # X: N x D matrix of training inputs
    # y: N x 1 vector of training targets/observations
    # returns: maximum likelihood parameters (D x 1)
    
    theta_ml = np.linalg.solve(X.T @ X, X.T @ y)
    return theta_ml

In [34]:
def lr_predict(Xtest, theta):
    
    # Xtest: K x D matrix of test inputs
    # theta: D x 1 vector of parameters
    # returns: prediction of f(Xtest); K x 1 vector
    
    prediction =  Xtest @ theta
    
    return prediction 

## Successor State Predictions

Get training data (i.e. multiple trajectories)

In [35]:
num_trajectories = 5
traj = []
states = []
for j in tqdm(range(num_trajectories)):
    
    x = np.array([np.pi/2,0]) #np.random.uniform([-np.pi/2,-1],[np.pi/2,1]) # pendulum initialised at left position with zero angular vel 
    T = 50

    for i in range(T):
        u = np.random.uniform(-2,2)
        x_old = x
        X_aug = np.hstack((math.cos(x[0]),math.sin(x[0]),x[1],u))  # augment 1 for biases here
        
        
        x,_,_,_ = step(x,u)
        states.append([x_old[0],u,x[0]])

        # preprocess successor state (rect -> polar)
        y = np.array([x[0]-x_old[0],x[1]-x_old[1]])

        # predict, just the state (not the angular velocity)
        traj.append(np.hstack((X_aug,y)))    
        animate(x,j+i)

  0%|          | 0/5 [00:00<?, ?it/s]

Plot targets $y$

In [36]:
viz_states = np.array(states)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(viz_states[:,0],viz_states[:,1],viz_states[:,2])
plt.show()

## Train-test split

In [46]:
D = len(X_aug)
X = np.array(traj)[:,:D]
y = np.array(traj)[:,D:]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,shuffle=True) # this also shuffles the data

In [47]:
x_in = X_train[:T,:2]
y_ = X_train[1:T+1,:2] - x_in

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x_in[:,0],x_in[:,1],y_[:,0])
plt.show()

ValueError: operands could not be broadcast together with shapes (159,2) (160,2) 

In [179]:
plt.close()
plt.plot(y)
plt.legend(["$\Theta$","$\dot{\Theta}$"])
plt.show()
plt.axis("tight")

(-0.05, 1.05, 4.045599641587482, 7.621544352376926)

## Fit model from training trajectories

In [48]:
lrmle_theta = lr_ml_estimate(X_train, y_train)

Get test set predictions and get losses

In [49]:
y_train_preds = lr_predict(X_train,lrmle_theta)
y_test_preds = lr_predict(X_test, lrmle_theta)

In [50]:
# train MSE
mse(y_train,y_train_preds)

3.7821680193560843

In [51]:
# test MSE
mse(y_test[:,1],y_test_preds[:,1])

6.85631060201856e-31

In [43]:
plt.plot(y_test[:,1])
plt.plot(y_test_preds[:,1])
plt.legend(["test","preds"])
plt.show()


In [44]:
x = np.array([np.pi/2,0]) # pendulum initialised at left position with zero angular vel 
T = 200
traj = []
losses = []

for i in tqdm(range(T)):
    u = np.random.uniform(-2,2)
    X_aug = np.hstack((math.cos(x[0]),math.sin(x[0]),x[1],u)) # augment 1 for biases here
    
    x_old = x
    # predict next state
    y_pred = lr_predict(X_aug, lrmle_theta) + x_old
    
    x,_,_,_ = step(x,u)
    
    # get actual successor state
    y = np.array([x[0],x[1]])
    
    # compute and save mse loss
    losses.append(mse(np.array([y[0]]),np.array([y_pred[0]])))
    
    traj.append(np.hstack((X_aug,y)))
    
    # animate(x)

  0%|          | 0/200 [00:00<?, ?it/s]

In [45]:
plt.close()
plt.plot(losses)
plt.show()
plt.xlabel("t")
plt.ylabel("Loss")
plt.axis("tight")

(-9.950000000000001, 208.95, -3.944304526105059e-32, 8.283039504820624e-31)

In [5]:
import torch

ModuleNotFoundError: No module named 'torch'